# Runner: Colab / Kaggle
Thin launcher only: **no logic lives here** (it is all in `src/` and `scripts/`). CPU runtime is enough, no GPU needed.
Edit the **CONFIG** cell once, then run top to bottom. Each stage is a separate cell so you can run only yours.

In [ ]:
# ---------- CONFIG (edit) ----------
GITHUB_REPO = "YOUR_ORG_OR_USER/YOUR_REPO"        # e.g. "hamna/tomato-robust-ml"
GIT_USER, GIT_EMAIL = "Your Name", "you@example.com"
DRIVE_DATA   = "/content/drive/MyDrive/ML_Project/tomato_dataset"   # Colab: folder with the raw images (or the unzipped dataset)
DRIVE_BACKUP = "/content/drive/MyDrive/ML_Project/work_backup"      # Colab: where work/ (cache, features, models) is backed up
KAGGLE_DATA  = "/kaggle/input/YOUR-DATASET-SLUG"                    # Kaggle: path of the attached dataset
# -----------------------------------

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")       # Kaggle: Add-ons > Secrets; Settings > Internet ON
    BASE = "/kaggle/working"; DATA = KAGGLE_DATA
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GH_TOKEN")                         # Colab: key icon (Secrets) in the left bar
    BASE = "/content"; DATA = DRIVE_DATA
REPO = f"{BASE}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_REPO}.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--rebase"], check=True)
os.chdir(REPO)
subprocess.run(["git", "config", "user.name", GIT_USER]); subprocess.run(["git", "config", "user.email", GIT_EMAIL])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Big files go to LOCAL disk (fast); results go inside the repo (small, committed).
os.environ["TOMATO_DATA_ROOT"]   = DATA
os.environ["TOMATO_WORK_DIR"]    = f"{BASE}/work"
os.environ["TOMATO_RESULTS_DIR"] = f"{REPO}/results"
print("repo:", REPO, "| data:", DATA, "| cpus:", os.cpu_count())

## Restore previous work (Colab only): copy cache/features/models back from Drive
Colab wipes local disk when a session ends. Skip this cell the very first time.

In [ ]:
if not ON_KAGGLE and os.path.exists(DRIVE_BACKUP):
    !mkdir -p /content/work && cp -r {DRIVE_BACKUP}/. /content/work/
    print("restored work/ from Drive")

## 01 Prepare data (Member A): manifest + image cache
Run once; commit `data/manifest.csv`. If the dataset is a zip on Drive, unzip it to `/content/data` first and point `DRIVE_DATA` there (much faster than reading thousands of files off Drive).

In [ ]:
!python scripts/01_prepare_data.py

## 02 EDA, segmentation check, duplicates (Member A)

In [ ]:
!python scripts/02_eda_duplicates.py

## 03 Features (Member B)
Benchmark first, then the full extraction.

In [ ]:
!python scripts/03_extract_features.py --benchmark 100

In [ ]:
!python scripts/03_extract_features.py

## 04 Baseline (RF / SVM / LR)  ·  05 Ablation (Member B)

In [ ]:
!python scripts/04_baseline.py

In [ ]:
!python scripts/05_ablation.py

## 06 Robustness  ·  07 Importance stability  ·  08 Error analysis  ·  09 Figures (Member C)

In [ ]:
!python scripts/06_robustness.py

In [ ]:
!python scripts/07_importance_stability.py

In [ ]:
!python scripts/08_error_analysis.py

In [ ]:
!python scripts/09_figures.py

## Save: back up `work/` (Colab) and push small results to GitHub
Push to **your own branch**, then open a pull request.

In [ ]:
if not ON_KAGGLE:
    !mkdir -p {DRIVE_BACKUP} && cp -r /content/work/. {DRIVE_BACKUP}/
BRANCH = "a/data"          # <-- your branch: a/..., b/..., c/...
!git checkout -B {BRANCH}
!git add results data/manifest.csv
!git commit -m "results: update from notebook run" || echo "nothing to commit"
!git push -u origin {BRANCH}